# EEG Pipeline v2 — Cleanup & Publication Figures
Works with v2 CSV files. No need to re-download raw data.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('whitegrid')

# Upload v2_all_epochs.csv, v2_all_resting.csv, v2_nri_sessions.csv
# from your Google Drive or local files

df = pd.read_csv('v2_all_epochs.csv')
df_rest = pd.read_csv('v2_all_resting.csv')
df_nri_raw = pd.read_csv('v2_nri_sessions.csv')

print(f'Task epochs: {len(df)}')
print(f'Resting epochs: {len(df_rest)}')
print(f'Subjects: {df["subject"].nunique()}')

## Step 1: Outlier Removal
ROI α/β ratio has extreme values (up to 70) when frontal beta ≈ 0 after ICA.
Strategy: winsorize at 1st and 99th percentile per metric.

In [ ]:
# Before cleanup
print('BEFORE CLEANUP:')
for col in ['roi_ab_ratio', 'theta_beta_ratio', 'global_ab_ratio']:
    print(f'  {col}: mean={df[col].mean():.2f}, median={df[col].median():.2f}, '
          f'max={df[col].max():.1f}, 99th={df[col].quantile(0.99):.2f}')

# Winsorize: clip extreme values to 1st/99th percentile
metrics_to_clip = ['roi_ab_ratio', 'theta_beta_ratio', 'global_ab_ratio',
                   'frontal_theta', 'parietal_alpha', 'frontal_beta']

for col in metrics_to_clip:
    if col in df.columns:
        lo = df[col].quantile(0.01)
        hi = df[col].quantile(0.99)
        df[col] = df[col].clip(lo, hi)
    if col in df_rest.columns:
        lo = df_rest[col].quantile(0.01)
        hi = df_rest[col].quantile(0.99)
        df_rest[col] = df_rest[col].clip(lo, hi)

print('\nAFTER CLEANUP:')
for col in ['roi_ab_ratio', 'theta_beta_ratio', 'global_ab_ratio']:
    print(f'  {col}: mean={df[col].mean():.2f}, median={df[col].median():.2f}, '
          f'max={df[col].max():.1f}, 99th={df[col].quantile(0.99):.2f}')

# Cognitive tasks only
df_cog = df[df['workload'].isin(['Low', 'Medium', 'High'])].copy()
n_subjects = df_cog['subject'].nunique()
print(f'\nCognitive task epochs: {len(df_cog)}, Subjects: {n_subjects}')

## Step 2: Recalculate NRI with cleaned data

In [ ]:
# Recompute NRI from cleaned resting data
nri_results = []

for sub in df_rest['subject'].unique():
    for ses in df_rest[df_rest['subject']==sub]['session'].unique():
        sub_ses = df_rest[(df_rest['subject']==sub) & (df_rest['session']==ses)]
        
        pre_ec = sub_ses[sub_ses['resting_type'] == 'pre_EC']['roi_ab_ratio']
        post_ec = sub_ses[sub_ses['resting_type'] == 'post_EC']['roi_ab_ratio']
        
        if len(pre_ec) > 0 and len(post_ec) > 0:
            pre_mean = pre_ec.mean()
            post_mean = post_ec.mean()
            nri = post_mean - pre_mean
            nri_norm = post_mean / pre_mean if pre_mean > 0 else 0
            nri_results.append({
                'subject': sub, 'session': ses,
                'pre_ab_ratio': pre_mean, 'post_ab_ratio': post_mean,
                'nri': nri, 'nri_normalized': nri_norm
            })

df_nri = pd.DataFrame(nri_results)
nri_by_sub = df_nri.groupby('subject').agg(
    nri_mean=('nri', 'mean'), nri_std=('nri', 'std'),
    nri_norm_mean=('nri_normalized', 'mean'), n_sessions=('session', 'count')
).reset_index()
nri_by_sub['recovery_group'] = nri_by_sub['nri_mean'].apply(
    lambda x: 'High Recovery' if x > 0 else 'Low Recovery')

print(f'Mean NRI: {nri_by_sub["nri_mean"].mean():.3f} ± {nri_by_sub["nri_mean"].std():.3f}')
print(f'Recovery groups:')
print(nri_by_sub['recovery_group'].value_counts())

## Step 3: Full Statistics (cleaned)

In [ ]:
def full_stats(data, metric, label):
    print(f'\n{"="*65}')
    print(f'{label}')
    print(f'{"="*65}')
    groups = {wl: data[data['workload']==wl][metric] for wl in ['Low','Medium','High']}
    
    for wl in ['Low','Medium','High']:
        g = groups[wl]
        print(f'  {wl:8s}: {g.mean():.3f} ± {g.std():.3f} (median={g.median():.3f}, n={len(g)})')
    
    f_stat, p_val = stats.f_oneway(*groups.values())
    print(f'\n  ANOVA: F={f_stat:.3f}, p={p_val:.2e} {"✓" if p_val < 0.05 else "✗"}')
    
    for n1, n2 in [('Low','Medium'), ('Medium','High'), ('Low','High')]:
        t, p = stats.ttest_ind(groups[n1], groups[n2])
        n_1, n_2 = len(groups[n1]), len(groups[n2])
        pooled = np.sqrt(((n_1-1)*groups[n1].var() + (n_2-1)*groups[n2].var()) / (n_1+n_2-2))
        d = (groups[n1].mean() - groups[n2].mean()) / pooled if pooled > 0 else 0
        size = 'small' if abs(d) < 0.5 else 'medium' if abs(d) < 0.8 else 'large'
        sig = '✓' if p < 0.05 else '✗'
        print(f'  {n1:6s} vs {n2:6s}: t={t:7.3f}, p={p:.2e} {sig}, d={d:+.3f} ({size})')

full_stats(df_cog, 'roi_ab_ratio', 'ROI α/β RATIO (parietal α / frontal β)')
full_stats(df_cog, 'global_ab_ratio', 'GLOBAL α/β RATIO (old method)')
full_stats(df_cog, 'theta_beta_ratio', 'θ/β RATIO (frontal θ / frontal β)')
full_stats(df_cog, 'frontal_theta', 'FRONTAL MIDLINE THETA POWER')

In [ ]:
# Task-specific
print('='*65)
print('N-BACK')
print('='*65)
for fname, label in [('zeroBACK','0-back'), ('oneBACK','1-back'), ('twoBACK','2-back')]:
    d = df[df['filename']==fname]
    print(f'  {label}: ROI α/β={d["roi_ab_ratio"].mean():.2f}±{d["roi_ab_ratio"].std():.2f}  '
          f'θ/β={d["theta_beta_ratio"].mean():.2f}±{d["theta_beta_ratio"].std():.2f}  '
          f'θ={d["frontal_theta"].mean():.2e}  (n={len(d)})')

nback = df[df['filename'].isin(['zeroBACK','oneBACK','twoBACK'])]
for metric, name in [('roi_ab_ratio','ROI α/β'), ('theta_beta_ratio','θ/β'), ('frontal_theta','Frontal θ')]:
    groups = [nback[nback['filename']==f][metric] for f in ['zeroBACK','oneBACK','twoBACK']]
    f, p = stats.f_oneway(*groups)
    print(f'  ANOVA {name}: F={f:.3f}, p={p:.2e} {"✓" if p < 0.05 else "✗"}')

print(f'\n{"="*65}')
print('MATB')
print('='*65)
for fname, label in [('MATBeasy','Easy'), ('MATBmed','Medium'), ('MATBdiff','Difficult')]:
    d = df[df['filename']==fname]
    print(f'  {label}: ROI α/β={d["roi_ab_ratio"].mean():.2f}±{d["roi_ab_ratio"].std():.2f}  '
          f'θ/β={d["theta_beta_ratio"].mean():.2f}±{d["theta_beta_ratio"].std():.2f}  '
          f'θ={d["frontal_theta"].mean():.2e}  (n={len(d)})')

matb = df[df['filename'].isin(['MATBeasy','MATBmed','MATBdiff'])]
for metric, name in [('roi_ab_ratio','ROI α/β'), ('theta_beta_ratio','θ/β'), ('frontal_theta','Frontal θ')]:
    groups = [matb[matb['filename']==f][metric] for f in ['MATBeasy','MATBmed','MATBdiff']]
    f, p = stats.f_oneway(*groups)
    print(f'  ANOVA {name}: F={f:.3f}, p={p:.2e} {"✓" if p < 0.05 else "✗"}')

## Step 4: Publication-Ready Figures

In [ ]:
# ---- Common settings ----
order = ['Low', 'Medium', 'High']
palette = {'Low': '#4A90D9', 'Medium': '#50B86C', 'High': '#E05D5D'}
FIG_DPI = 300

def add_significance(ax, x1, x2, y, p, h=0.02):
    """Add significance bracket."""
    if p < 0.001: text = '***'
    elif p < 0.01: text = '**'
    elif p < 0.05: text = '*'
    else: text = 'ns'
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], color='black', linewidth=1)
    ax.text((x1+x2)/2, y+h, text, ha='center', va='bottom', fontsize=10)

In [ ]:
# ---- FIGURE 1: Main result — ROI α/β by Workload ----

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# A: Alpha proportion
sns.boxplot(data=df_cog, x='workload', y='roi_alpha_rel', order=order,
            palette=palette, ax=axes[0], fliersize=2, linewidth=1)
axes[0].axhline(y=0.667, color='#888', linestyle='--', alpha=0.6, label='Hypothesis (2/3)')
axes[0].set_title('(A) Parietal α proportion', fontsize=12, fontweight='bold')
axes[0].set_ylabel('α / (α + β)', fontsize=11)
axes[0].set_xlabel('Workload Level', fontsize=11)
axes[0].legend(fontsize=9)

# B: Beta proportion
sns.boxplot(data=df_cog, x='workload', y='roi_beta_rel', order=order,
            palette=palette, ax=axes[1], fliersize=2, linewidth=1)
axes[1].axhline(y=0.333, color='#888', linestyle='--', alpha=0.6, label='Hypothesis (1/3)')
axes[1].set_title('(B) Frontal β proportion', fontsize=12, fontweight='bold')
axes[1].set_ylabel('β / (α + β)', fontsize=11)
axes[1].set_xlabel('Workload Level', fontsize=11)
axes[1].legend(fontsize=9)

# C: ROI α/β ratio
sns.boxplot(data=df_cog, x='workload', y='roi_ab_ratio', order=order,
            palette=palette, ax=axes[2], fliersize=2, linewidth=1)
axes[2].axhspan(1.2, 2.0, alpha=0.08, color='green', label='Original flow zone')
axes[2].set_title('(C) ROI α/β ratio', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Parietal α / Frontal β', fontsize=11)
axes[2].set_xlabel('Workload Level', fontsize=11)
axes[2].legend(fontsize=9)

# Add significance brackets to panel C
ymax = df_cog['roi_ab_ratio'].quantile(0.99) * 1.05
groups_c = {wl: df_cog[df_cog['workload']==wl]['roi_ab_ratio'] for wl in order}
_, p_lm = stats.ttest_ind(groups_c['Low'], groups_c['Medium'])
_, p_mh = stats.ttest_ind(groups_c['Medium'], groups_c['High'])
_, p_lh = stats.ttest_ind(groups_c['Low'], groups_c['High'])
add_significance(axes[2], 0, 1, ymax*0.88, p_lm)
add_significance(axes[2], 1, 2, ymax*0.93, p_mh)
add_significance(axes[2], 0, 2, ymax*0.99, p_lh)

plt.suptitle(f'EEG Band Power by Workload — ROI Analysis (N={n_subjects})',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('fig1_roi_workload.png', dpi=FIG_DPI, bbox_inches='tight')
plt.savefig('fig1_roi_workload.pdf', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ---- FIGURE 2: N-Back and MATB ----

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# A: N-Back
nback = df[df['task'] == 'N-Back']
sns.boxplot(data=nback, x='filename', y='roi_ab_ratio',
            order=['zeroBACK', 'oneBACK', 'twoBACK'],
            palette=['#4A90D9', '#50B86C', '#E05D5D'], ax=axes[0],
            fliersize=2, linewidth=1)
axes[0].set_title('(A) N-Back: ROI α/β by difficulty', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(['0-back\n(Low)', '1-back\n(Medium)', '2-back\n(High)'])
axes[0].set_ylabel('ROI α/β Ratio', fontsize=11)
axes[0].set_xlabel('')

# Add significance
nback_groups = [nback[nback['filename']==f]['roi_ab_ratio'] for f in ['zeroBACK','oneBACK','twoBACK']]
ymax_n = nback['roi_ab_ratio'].quantile(0.99) * 1.05
_, p01 = stats.ttest_ind(nback_groups[0], nback_groups[1])
_, p12 = stats.ttest_ind(nback_groups[1], nback_groups[2])
_, p02 = stats.ttest_ind(nback_groups[0], nback_groups[2])
add_significance(axes[0], 0, 2, ymax_n*0.95, p02)

# B: MATB
matb = df[df['task'] == 'MATB']
sns.boxplot(data=matb, x='filename', y='roi_ab_ratio',
            order=['MATBeasy', 'MATBmed', 'MATBdiff'],
            palette=['#4A90D9', '#50B86C', '#E05D5D'], ax=axes[1],
            fliersize=2, linewidth=1)
axes[1].set_title('(B) MATB: ROI α/β by difficulty', fontsize=12, fontweight='bold')
axes[1].set_xticklabels(['Easy\n(Low)', 'Medium', 'Difficult\n(High)'])
axes[1].set_ylabel('ROI α/β Ratio', fontsize=11)
axes[1].set_xlabel('')

matb_groups = [matb[matb['filename']==f]['roi_ab_ratio'] for f in ['MATBeasy','MATBmed','MATBdiff']]
_, p_matb = stats.f_oneway(*matb_groups)
axes[1].text(0.95, 0.95, f'ANOVA: p={p_matb:.3f} (ns)',
             transform=axes[1].transAxes, ha='right', va='top', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle(f'Task-Specific Workload Analysis (N={n_subjects})',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('fig2_nback_matb.png', dpi=FIG_DPI, bbox_inches='tight')
plt.savefig('fig2_nback_matb.pdf', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ---- FIGURE 3: Frontal Theta Analysis ----

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# A: Frontal theta by workload
sns.boxplot(data=df_cog, x='workload', y='frontal_theta', order=order,
            palette=palette, ax=axes[0], fliersize=2, linewidth=1)
axes[0].set_title('(A) Frontal midline θ power', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Frontal θ Power (V²/Hz)', fontsize=11)
axes[0].set_xlabel('Workload Level', fontsize=11)

# Add significance
theta_groups = {wl: df_cog[df_cog['workload']==wl]['frontal_theta'] for wl in order}
ymax_t = df_cog['frontal_theta'].quantile(0.99) * 1.05
_, p_t_lh = stats.ttest_ind(theta_groups['Low'], theta_groups['High'])
_, p_t_mh = stats.ttest_ind(theta_groups['Medium'], theta_groups['High'])
add_significance(axes[0], 0, 2, ymax_t*0.95, p_t_lh)
add_significance(axes[0], 1, 2, ymax_t*0.87, p_t_mh)

# B: θ/β ratio by workload
sns.boxplot(data=df_cog, x='workload', y='theta_beta_ratio', order=order,
            palette=palette, ax=axes[1], fliersize=2, linewidth=1)
axes[1].set_title('(B) θ/β ratio by workload', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Frontal θ / Frontal β', fontsize=11)
axes[1].set_xlabel('Workload Level', fontsize=11)

# C: N-Back theta
nback_cog = df[df['filename'].isin(['zeroBACK','oneBACK','twoBACK'])]
sns.boxplot(data=nback_cog, x='filename', y='frontal_theta',
            order=['zeroBACK','oneBACK','twoBACK'],
            palette=['#4A90D9','#50B86C','#E05D5D'], ax=axes[2],
            fliersize=2, linewidth=1)
axes[2].set_title('(C) N-Back: frontal θ by difficulty', fontsize=12, fontweight='bold')
axes[2].set_xticklabels(['0-back\n(Low)', '1-back\n(Med)', '2-back\n(High)'])
axes[2].set_ylabel('Frontal θ Power', fontsize=11)
axes[2].set_xlabel('')

plt.suptitle(f'Frontal Theta Analysis (N={n_subjects})',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('fig3_theta.png', dpi=FIG_DPI, bbox_inches='tight')
plt.savefig('fig3_theta.pdf', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ---- FIGURE 4: Neural Recovery Index ----

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# A: Pre vs Post
pre_vals = df_nri['pre_ab_ratio']
post_vals = df_nri['post_ab_ratio']
bp_data = pd.DataFrame({
    'Condition': ['Pre-session']*len(pre_vals) + ['Post-session']*len(post_vals),
    'ROI α/β': list(pre_vals) + list(post_vals)
})
sns.boxplot(data=bp_data, x='Condition', y='ROI α/β',
            palette=['#4A90D9','#E05D5D'], ax=axes[0], fliersize=2, linewidth=1)
for _, row in df_nri.iterrows():
    axes[0].plot([0, 1], [row['pre_ab_ratio'], row['post_ab_ratio']],
                 color='gray', alpha=0.15, linewidth=0.5)
axes[0].set_title('(A) Resting ROI α/β: pre vs post', fontsize=12, fontweight='bold')
axes[0].set_ylabel('ROI α/β (Eyes Closed)', fontsize=11)
axes[0].set_xlabel('')

# B: NRI distribution
axes[1].hist(nri_by_sub['nri_mean'], bins=12, color='#50B86C', edgecolor='white', alpha=0.8)
axes[1].axvline(x=0, color='#E05D5D', linestyle='--', linewidth=1.5, label='No recovery')
mean_nri = nri_by_sub['nri_mean'].mean()
axes[1].axvline(x=mean_nri, color='#2c3e50', linestyle='-', linewidth=1.5,
                label=f'Mean NRI={mean_nri:.2f}')
axes[1].set_title('(B) NRI distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('NRI (post − pre ROI α/β)', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].legend(fontsize=9)

# C: NRI by subject
nri_sorted = nri_by_sub.sort_values('nri_mean')
colors = ['#50B86C' if x > 0 else '#E05D5D' for x in nri_sorted['nri_mean']]
axes[2].barh(range(len(nri_sorted)), nri_sorted['nri_mean'], color=colors, edgecolor='white')
axes[2].set_yticks(range(len(nri_sorted)))
axes[2].set_yticklabels(nri_sorted['subject'], fontsize=8)
axes[2].axvline(x=0, color='black', linewidth=0.5)
axes[2].set_title('(C) NRI by subject', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Mean NRI', fontsize=11)

plt.suptitle(f'Neural Recovery Index (N={n_subjects})',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('fig4_nri.png', dpi=FIG_DPI, bbox_inches='tight')
plt.savefig('fig4_nri.pdf', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ---- FIGURE 5: Inter-subject variability + ROI vs Global comparison ----

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# A: Subject-level ROI α/β
sub_means = df_cog.groupby(['subject','workload'])['roi_ab_ratio'].mean().reset_index()
sns.stripplot(data=sub_means, x='workload', y='roi_ab_ratio', order=order,
              palette=palette, alpha=0.7, jitter=0.2, ax=axes[0], size=6)
sns.boxplot(data=sub_means, x='workload', y='roi_ab_ratio', order=order,
            palette=palette, boxprops=dict(alpha=0.25), ax=axes[0],
            fliersize=0, linewidth=1)
axes[0].set_title('(A) Subject-level ROI α/β', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Mean ROI α/β per Subject', fontsize=11)
axes[0].set_xlabel('Workload Level', fontsize=11)

# B: ROI vs Global — paired comparison of F-statistics
comparison_data = []
for metric, name in [('roi_ab_ratio','ROI α/β'), ('global_ab_ratio','Global α/β'),
                      ('theta_beta_ratio','θ/β'), ('frontal_theta','Frontal θ')]:
    groups = [df_cog[df_cog['workload']==wl][metric] for wl in order]
    f_stat, p_val = stats.f_oneway(*groups)
    comparison_data.append({'Metric': name, 'F-statistic': f_stat, 'p-value': p_val})

comp_df = pd.DataFrame(comparison_data)
bars = axes[1].barh(range(len(comp_df)), comp_df['F-statistic'],
                     color=['#2c3e50','#7f8c8d','#8e44ad','#e67e22'])
axes[1].set_yticks(range(len(comp_df)))
axes[1].set_yticklabels(comp_df['Metric'], fontsize=11)
axes[1].set_xlabel('F-statistic (ANOVA)', fontsize=11)
axes[1].set_title('(B) Metric comparison: workload discrimination', fontsize=12, fontweight='bold')

for i, row in comp_df.iterrows():
    p_text = f'p={row["p-value"]:.1e}' if row['p-value'] < 0.001 else f'p={row["p-value"]:.3f}'
    axes[1].text(row['F-statistic']+1, i, p_text, va='center', fontsize=9)

plt.suptitle(f'Inter-Subject Variability & Metric Comparison (N={n_subjects})',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('fig5_variability.png', dpi=FIG_DPI, bbox_inches='tight')
plt.savefig('fig5_variability.pdf', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

## Step 5: Summary Table for Paper

In [ ]:
# Generate LaTeX-ready summary table
print('\n% ---- TABLE: Summary Statistics by Workload ----')
print('% Paste into your LaTeX document')
print()
print(r'\begin{table}[htbp]')
print(r'  \centering')
print(r'  \caption{EEG Spectral Metrics by Workload Level (N=20, ICA-cleaned, ROI-based)}')
print(r'  \label{tab:roi_results}')
print(r'  \begin{tabular}{lcccc}')
print(r'    \toprule')
print(r'    \textbf{Metric} & \textbf{Low} & \textbf{Medium} & \textbf{High} & \textbf{ANOVA} \\')
print(r'    \midrule')

for metric, label in [('roi_ab_ratio', r'ROI $\alpha/\beta$'),
                       ('global_ab_ratio', r'Global $\alpha/\beta$'),
                       ('theta_beta_ratio', r'$\theta/\beta$'),
                       ('frontal_theta', r'Frontal $\theta$')]:
    vals = []
    for wl in ['Low','Medium','High']:
        g = df_cog[df_cog['workload']==wl][metric]
        if metric == 'frontal_theta':
            vals.append(f'{g.mean():.2e}')
        else:
            vals.append(f'{g.mean():.2f} $\\pm$ {g.std():.2f}')
    
    groups = [df_cog[df_cog['workload']==wl][metric] for wl in ['Low','Medium','High']]
    f_stat, p_val = stats.f_oneway(*groups)
    if p_val < 0.001:
        p_str = f'$F$={f_stat:.1f}, $p$<0.001'
    else:
        p_str = f'$F$={f_stat:.1f}, $p$={p_val:.3f}'
    
    print(f'    {label} & {vals[0]} & {vals[1]} & {vals[2]} & {p_str} \\\\')

print(r'    \bottomrule')
print(r'  \end{tabular}')
print(r'\end{table}')

print('\n\n% ---- TABLE: N-Back Results ----')
print(r'\begin{table}[htbp]')
print(r'  \centering')
print(r'  \caption{N-Back Task: ROI Metrics by Difficulty Level}')
print(r'  \label{tab:nback}')
print(r'  \begin{tabular}{lccc}')
print(r'    \toprule')
print(r'    \textbf{Level} & \textbf{ROI $\alpha/\beta$} & \textbf{$\theta/\beta$} & \textbf{Frontal $\theta$} \\')
print(r'    \midrule')

for fname, label in [('zeroBACK', '0-back (Low)'), ('oneBACK', '1-back (Medium)'), ('twoBACK', '2-back (High)')]:
    d = df[df['filename']==fname]
    print(f'    {label} & {d["roi_ab_ratio"].mean():.2f} $\\pm$ {d["roi_ab_ratio"].std():.2f}'
          f' & {d["theta_beta_ratio"].mean():.2f} $\\pm$ {d["theta_beta_ratio"].std():.2f}'
          f' & {d["frontal_theta"].mean():.2e} \\\\\\\\')

print(r'    \bottomrule')
print(r'  \end{tabular}')
print(r'\end{table}')

print('\nDone! Copy tables into your LaTeX document.')

In [ ]:
# Save all cleaned data
df_cog.to_csv('v2_clean_epochs.csv', index=False)
nri_by_sub.to_csv('v2_clean_nri.csv', index=False)

print('Files saved:')
print('  Figures: fig1-5 (.png + .pdf)')
print('  Data: v2_clean_epochs.csv, v2_clean_nri.csv')
print('\nReady for paper!')